<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/02%20-%20Supervised%20Learning/02c_classification.ipynb)

# 02c — 分类与基础特征重要性（Classification & Feature Importance）

在 `02a` 和 `02b` 中，我们用回归模型预测了具体的 $\log S$ 数值。但在很多实验室场景中，研究人员并不需要精确到小数点后两位的溶解度，而是关心一个更直接的**高通量筛查（High-Throughput Screening）**问题：

> **“这批候选分子里，哪些几乎不溶（难加工，趁早淘汰），哪些能溶（值得合成与测试）？”**

当预测目标从连续数值变成离散的类别标签（“难溶” vs “可溶”、“有毒” vs “无毒”、“金属” vs “绝缘体”）时，任务就从**回归（Regression）**变成了**分类（Classification）**。

本讲包含两部分核心内容：
1. **二元分类与评估体系**：如何定义类别标签、为什么准确率（Accuracy）在类别不平衡时会失灵，以及如何借助混淆矩阵、精确率/召回率（Precision/Recall）、ROC 曲线和决策阈值（Threshold）定制符合实验需求的筛查策略；
2. **基础特征重要性（Feature Importance）**：训练好随机森林后，打开“黑箱”看看模型到底依赖哪些分子描述符做判断，并对比模型学到的规律与物理化学先验的一致性。

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "02 - Supervised Learning"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


---

## 0. 环境准备与数据读入

我们继续使用 ESOL 数据集，并用与 `02a`、`02b` 完全相同的流程计算 10 个 RDKit 物理化学描述符作为输入特征矩阵 `X`。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
)
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


In [ ]:
DATA_PATHS = [Path("../01 - Basics/data/esol.csv"), Path("data/esol.csv")]
for p in DATA_PATHS:
    if p.exists():
        df = pd.read_csv(p)
        break
df = df.rename(columns={
    "Compound ID": "name",
    "measured log solubility in mols per litre": "logS",
})[["name", "smiles", "logS"]].copy()
df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)
df = df.dropna(subset=["mol"]).reset_index(drop=True)

DESC_FUNCS = {
    "MolWt": Descriptors.MolWt,
    "MolLogP": Descriptors.MolLogP,
    "TPSA": Descriptors.TPSA,
    "NumHDonors": Descriptors.NumHDonors,
    "NumHAcceptors": Descriptors.NumHAcceptors,
    "NumRotatableBonds": Descriptors.NumRotatableBonds,
    "RingCount": Descriptors.RingCount,
    "NumAromaticRings": Descriptors.NumAromaticRings,
    "HeavyAtomCount": Descriptors.HeavyAtomCount,
    "FractionCSP3": Descriptors.FractionCSP3,
}
X = pd.DataFrame(
    [{k: fn(m) for k, fn in DESC_FUNCS.items()} for m in df["mol"]]
)

---

## 1. 从连续回归到二元分类：定义“难溶分子”筛查任务

在药物化学与有机电子溶液加工的经验规则中，水溶解度 $\log S < -4$（即摩尔浓度低于 $10^{-4}\text{ mol/L}$，对分子量 300 的化合物约合 $30\text{ mg/L}$）通常被视为**难溶（insoluble / poorly soluble）**的警戒线。

如果我们的任务是**把难溶分子揪出来淘汰掉**，就可以将我们重点关注的“难溶”定义为**正类（Positive Class，标签记为 `1`）**，将“可溶”定义为**负类（Negative Class，标签记为 `0`）**：

$$y = \begin{cases} 1 & \text{if } \log S < -4 \quad (\text{insoluble，难溶，正类}) \\ 0 & \text{if } \log S \ge -4 \quad (\text{soluble，可溶，负类}) \end{cases}$$

> **关于“正类（1）”的命名习惯**：机器学习里的“正类（Positive）”并不代表性质好坏，而是指**“我们要检测/报警的目标事件”**——就像医学检测中把“患病”记为阳性（Positive = 1）、质量检测中把“缺陷品”记为 1 一样。这里我们要筛查并拦截难溶分子，因此把难溶记为 `1`。

In [ ]:
THRESHOLD = -4.0
y = (df["logS"] < THRESHOLD).astype(int)   # 1 = insoluble, 0 = soluble

counts = y.value_counts().rename({0: "0: soluble (logS >= -4)", 1: "1: insoluble (logS < -4)"})
print(counts)
print(f"\n难溶分子（正类 1）占比: {y.mean():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(df.loc[y == 0, "logS"], bins=25, alpha=0.75, label="0: soluble (logS >= -4)", color="steelblue")
ax.hist(df.loc[y == 1, "logS"], bins=25, alpha=0.75, label="1: insoluble (logS < -4)", color="indianred")
ax.axvline(THRESHOLD, color="black", ls="--", lw=1.5, label="Cutoff = -4")
ax.set_xlabel("Measured logS (mol/L)")
ax.set_ylabel("Count")
ax.set_title("Binary Classification Target defined from logS")
ax.legend()
plt.tight_layout()
plt.show()

---

## 2. 分层抽样划分与分类器的两种输出

### 2.1 分层抽样（Stratified Split）

在 ESOL 数据集中，难溶分子占 `28.6%`，可溶分子占 `71.4%`。划分训练集和测试集时，如果在 `train_test_split` 中加上 `stratify=y`（分层抽样），scikit-learn 会保证训练集和测试集中正负类的比例与全集严格一致，避免因随机抽样导致测试集里正类偏多或偏少。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"训练集正类占比: {y_train.mean():.3f} | 测试集正类占比: {y_test.mean():.3f}")

### 2.2 训练分类器：`predict` 与 `predict_proba` 的区别

与回归器对应，scikit-learn 提供了同名的分类器（如 `RandomForestClassifier`、`KNeighborsClassifier`）。其中最经典的线性分类模型叫做**逻辑回归（`LogisticRegression`）**——注意它虽然名字里带“回归”，但实际上是通过 Sigmoid 函数 $\sigma(z) = \frac{1}{1 + e^{-z}}$ 将线性打分压缩到 $(0, 1)$ 区间、输出属于正类概率的**分类算法**。

训练好分类器后，有两种预测方式：
- `clf.predict(X_test)`：直接返回非黑即白的类别判断（`0` 或 `1`），默认以概率 `0.5` 为分界线；
- `clf.predict_proba(X_test)`：返回每个样本属于类别 `0` 和类别 `1` 的估计概率 `[P(y=0), P(y=1)]`。我们在做风险控制和阈值调节时，通常取第二列 `predict_proba(X_test)[:, 1]`（即判定为难溶的概率）。

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf_clf.fit(X_train, y_train)

logreg_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg_clf.fit(X_train, y_train)

y_pred_rf = rf_clf.predict(X_test)
y_prob_rf = rf_clf.predict_proba(X_test)[:, 1]   # 属于正类 1（难溶）的概率

pd.DataFrame({
    "True_y": y_test.values[:6],
    "Pred_y (cutoff=0.5)": y_pred_rf[:6],
    "P(insoluble)": y_prob_rf[:6].round(3),
})

---

## 3. 全面评估分类模型：为什么不能只看准确率（Accuracy）？

### 3.1 准确率（Accuracy）与“全猜多数类”的偷懒陷阱

评估分类模型时，最直观的指标是**准确率（Accuracy）**：猜对的样本占总样本的比例。

但请注意：我们的测试集中有 `71.4%` 的分子本来就是可溶的（`0`）。如果写一个完全不看分子特征、闭着眼睛把所有分子都猜成“可溶（`0`）”的傻瓜分类器（`DummyClassifier(strategy="most_frequent")`），它的准确率能拿多少？

In [ ]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)
y_pred_dummy = dummy_clf.predict(X_test)

print(f"Dummy (永远猜 0)  Accuracy: {accuracy_score(y_test, y_pred_dummy):.3f}")
print(f"LogisticRegression Accuracy: {accuracy_score(y_test, logreg_clf.predict(X_test)):.3f}")
print(f"RandomForest       Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")

哪怕一个难溶分子都没抓出来，傻瓜基线也能拿到 **71.5%** 的准确率！如果某个稀有缺陷或高活性分子在数据库中只占 `1%`，全猜 `0` 的准确率甚至高达 `99%`。

因此，在分类任务（尤其是正负类比例不均衡的科学数据集）中，**绝不能只汇报 Accuracy**。我们需要把“怎么猜对的、怎么猜错的”拆开来看。

### 3.2 混淆矩阵（Confusion Matrix）：四种预测结局

把真实标签（行）与模型预测标签（列）交叉列表，就得到了 2×2 的**混淆矩阵（Confusion Matrix）**：

| | 预测为 `0`（可溶） | 预测为 `1`（难溶） |
|---|---|---|
| **真实为 `0`（可溶）** | **TN（True Negative，真阴性）**<br>本来可溶，模型也放行 | **FP（False Positive，假阳性 / 误报）**<br>本来可溶的好分子，被模型冤枉淘汰了 |
| **真实为 `1`（难溶）** | **FN（False Negative，假阴性 / 漏报）**<br>本来难溶的坏分子，漏网进入后续实验 | **TP（True Positive，真阳性）**<br>本来难溶，被模型成功拦截 |

我们把 Dummy 基线与随机森林的混淆矩阵并排画出来：

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))

for ax, (name, preds) in zip(axes, [("Dummy (Always 0)", y_pred_dummy), ("Random Forest", y_pred_rf)]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, preds,
        display_labels=["Soluble (0)", "Insoluble (1)"],
        cmap="Blues", ax=ax, colorbar=False,
    )
    ax.set_title(name)
    ax.grid(False)

plt.tight_layout()
plt.show()

### 3.3 精确率（Precision）、召回率（Recall）、F1 与 ROC-AUC

从混淆矩阵的四个格子出发，可以定义三个核心分类指标：

1. **精确率（Precision，查准率）**：在所有**被模型报警为难溶**的分子中，真正难溶的比例是多少？
   $$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$
   - Precision 高意味着**“误报（FP）少”**，不会轻易冤枉一个可溶的好分子。
2. **召回率（Recall，查全率 / 灵敏度）**：在所有**实际上真正难溶**的分子中，有多大比例被模型成功揪出来了？
   $$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$
   - Recall 高意味着**“漏报（FN）少”**，很少有难溶分子能蒙混过关。
3. **F1 分数（F1-score）**：Precision 与 Recall 的调和平均数（只有两者都高时 F1 才会高）：
   $$\text{F1} = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

此外，上述三个指标都依赖于人为设定的概率分界线（默认 `0.5`）。如果想衡量模型输出的概率 `P(insoluble)` **在所有可能的阈值下整体把正负类排开的能力**，我们使用 **ROC-AUC**（ROC 曲线下面积）：
- `ROC-AUC = 1.0`：完美排序（所有正类的预测概率都高于负类）；
- `ROC-AUC = 0.5`：瞎猜水平（与抛硬币无异）。

In [ ]:
def eval_classifier(clf, X_te, y_te):
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]
    return {
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, y_prob),
    }

models = {
    "Dummy (most_frequent)": dummy_clf,
    "LogisticRegression": logreg_clf,
    "RandomForest": rf_clf,
}
pd.DataFrame({k: eval_classifier(m, X_test, y_test) for k, m in models.items()}).T.round(3)

对比表格立刻看清了真相：Dummy 基线的 Accuracy 虽然有 `0.715`，但它的 Recall 和 F1 都是 **0.000**，ROC-AUC 恰好是瞎猜的 **0.500**；而随机森林的 F1 达到 **0.83** 左右，ROC-AUC 高达 **0.955**。

我们把三个模型的 ROC 曲线画在同一张图上（曲线越向左上角拱起，面积 AUC 越大，模型区分正负类的能力越强）：

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 5))
for name, clf in models.items():
    RocCurveDisplay.from_estimator(clf, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random guess (AUC = 0.50)")
ax.set_title("ROC Curves on ESOL Test Set")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 🏋️ 练习 02c-1：加入 kNN 与梯度提升分类器对比

在 `models` 字典中再加入两个分类器，重新生成上面的五项指标对比表：
1. `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=10))`
2. `GradientBoostingClassifier(random_state=42)`

In [ ]:
# 在下方填写你的代码：


---

## 4. 类别不平衡与调节决策阈值（Threshold Tuning）

默认情况下，`clf.predict(X)` 把 `P(insoluble) >= 0.5` 的分子判为难溶。但在真实的实验流程中，**误报（FP）和漏报（FN）的代价往往是不对称的**：

- **场景 A（宁可错杀，不可放过）**：假设后续每合成并测试一个候选分子要花费两周时间和数千元试剂，而难溶分子一旦混进去就会导致整批旋涂成膜失败。此时我们极度厌恶**漏报（FN）**，希望**召回率（Recall）越高越好**——哪怕多拦截几个本来勉强能溶的分子也无所谓。
- **做法**：把报警阈值从 `0.5` **调低**到 `0.3` 甚至 `0.2`（只要模型觉得难溶概率超过 20% 就报警拦截）。

下面观察降低决策阈值时，Precision（查准率）与 Recall（查全率）此消彼长的权衡关系：

In [ ]:
rows = []
for th in [0.7, 0.5, 0.3, 0.2]:
    pred_th = (y_prob_rf >= th).astype(int)
    cm = confusion_matrix(y_test, pred_th)
    rows.append({
        "Threshold": th,
        "Precision": precision_score(y_test, pred_th),
        "Recall": recall_score(y_test, pred_th),
        "F1": f1_score(y_test, pred_th),
        "FN (漏报难溶分子数)": cm[1, 0],
        "FP (误杀可溶分子数)": cm[0, 1],
    })
pd.DataFrame(rows).set_index("Threshold").round(3)

可以看到：当阈值从 `0.5` 降到 `0.2` 时，测试集里的漏报分子数（FN）从 13 个锐减到 2 个（Recall 升至 `96.9%`），代价是误杀的可溶分子数（FP）从 9 个增加到了 24 个。**模型还是同一个模型，仅仅调节决策阈值，就能适配完全不同的实验成本诉求。**

### 🏋️ 练习 02c-2：极端难溶分子筛查（严重类别不平衡）

如果我们把筛查标准收紧为“极端难溶”：`y_extreme = (df["logS"] < -6.0).astype(int)`。
1. 计算全数据集中 `y_extreme == 1` 的占比（提示：大约只有 `11%`）；
2. 用 `stratify=y_extreme` 重新划分训练/测试集，分别训练 `DummyClassifier(strategy="most_frequent")` 和 `RandomForestClassifier(random_state=42)`，对比两者的 `Accuracy` 和 `F1`。

In [ ]:
# 在下方填写你的代码：


---

## 5. 实战筛查：有机半导体分子的溶解度分类与溶剂先验

现在我们把训练好的随机森林分类器 `rf_clf` 用到一组经典的共轭分子与侧链改性有机半导体分子上，看看模型给出的水相难溶概率 `P(insoluble)`：

In [ ]:
osc_Reload = [
    ("benzene",       "c1ccccc1"),
    ("naphthalene",   "c1ccc2ccccc2c1"),
    ("anthracene",    "c1ccc2cc3ccccc3cc2c1"),
    ("tetracene",     "c1ccc2cc3cc4ccccc4cc3cc2c1"),
    ("pentacene",     "c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1"),
    ("BTBT (core)",   "c1ccc2c(c1)sc3c2sc4ccccc34"),
    ("C8-BTBT",       "CCCCCCCCc1ccc2c(c1)sc3c2sc4cc(CCCCCCCC)ccc34"),
    ("thiophene",     "c1ccsc1"),
]
osc_mols = [Chem.MolFromSmiles(s) for _, s in osc_Reload]
X_osc = pd.DataFrame(
    [{k: fn(m) for k, fn in DESC_FUNCS.items()} for m in osc_mols],
    index=[name for name, _ in osc_Reload],
)
X_osc["P(insoluble in water)"] = rf_clf.predict_proba(X_osc)[:, 1].round(3)
X_osc["Pred_Class"] = np.where(X_osc["P(insoluble in water)"] >= 0.5, "Insoluble (1)", "Soluble (0)")
X_osc[["MolWt", "MolLogP", "NumAromaticRings", "FractionCSP3", "P(insoluble in water)", "Pred_Class"]]

观察上面的预测表，有一个非常值得材料系同学深思的现象：

- 在有机电子实验室里，未修饰的 `BTBT (core)` 刚性母核由于强烈的 $\pi\text{-}\pi$ 堆积而难溶于氯苯、甲苯等有机溶剂；合成化学家给它装上两条长烷基链变成 **`C8-BTBT`** 后，柔性脂肪链通过增加构象熵并打乱晶格堆积，使它在**有机溶剂**中的溶解度大幅提升，成为经典的溶液法旋涂小分子半导体。
- 但我们的模型却以 `1.000` 的概率判定 `C8-BTBT` **不溶**！为什么？

**原因在于训练集的任务定义**：ESOL 数据集测量的是**水溶解度（aqueous solubility）**。给 BTBT 加上两条长碳链（$2 \times \text{-C}_8\text{H}_{17}$）会让油水分配系数 `MolLogP` 从 `5.0` 暴涨到 `11.3`（极度疏水），在水里的溶解度自然变得更低。这个例子再次提醒我们：**机器学习模型永远只忠实于训练数据的物理条件（这里是水相），跨溶剂体系迁移时必须明确物理化学边界。**

---

## 6. 打开模型黑箱：模型到底在看什么特征？

在科研中，训练出一个高准确率的模型往往只是第一步；我们更关心：**模型是依据哪些物理化学特征做出判断的？它学到的规律符不符合化学直觉？**

下面介绍两种最常用的特征重要性（Feature Importance）分析方法，并揭示它们在面对**共线性特征（Collinear Features）**时的巨大差异。

### 6.1 随机森林自带的树分裂重要性（MDI / Gini Importance）

随机森林在训练每棵决策树时，每次挑选某个特征分裂节点都会带来不纯度（Gini Impurity）的下降。把一个特征在所有树上带来的不纯度下降加权平均，就得到了 `rf_clf.feature_importances_`（称为 **Mean Decrease in Impurity, MDI**）：

In [ ]:
mdi_imp = pd.Series(rf_clf.feature_importances_, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(6.5, 4))
mdi_imp.plot(kind="barh", color="steelblue", ax=ax)
ax.set_xlabel("Mean Decrease in Impurity (MDI)")
ax.set_title("Random Forest Built-in Feature Importance")
plt.tight_layout()
plt.show()

从 MDI 柱状图看，`MolLogP`（约 0.35）排第一，`MolWt`（分子量，约 0.19）和 `HeavyAtomCount`（重原子数，约 0.14）紧随其后。

但是请回想我们在 `01c` 画过的相关性热力图：**`MolWt` 和 `HeavyAtomCount` 的相关系数高达 `0.97`！** 当两个特征携带几乎相同的信息时，决策树在分裂节点时会随机二选一，导致它们各自的重要性被**平摊（稀释）**，同时挤占了其他特征的真实排名。

### 6.2 置换重要性（Permutation Importance）：更可靠的模型无关评估

为了克服树分裂重要性的偏差，我们可以直接在**测试集**上做如下思想实验（**置换重要性，Permutation Importance**）：

> 保持训练好的模型不变，把测试集中某一个特征列（比如 `MolLogP`）的数值**随机打乱顺序**（相当于毁掉该特征与分子的对应关系），然后看模型的测试集 ROC-AUC 掉了多少。掉得越多，说明模型在预测时越离不开这个特征。

In [ ]:
perm = permutation_importance(
    rf_clf, X_test, y_test,
    scoring="roc_auc", n_repeats=15, random_state=RANDOM_STATE, n_jobs=-1
)
perm_df = pd.DataFrame({
    "mean": perm.importances_mean,
    "std": perm.importances_std,
}, index=X.columns).sort_values("mean")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=False)

mdi_imp.plot(kind="barh", color="steelblue", ax=axes[0])
axes[0].set_title("1. Built-in MDI (Split among correlated features)")
axes[0].set_xlabel("MDI Importance")

axes[1].barh(perm_df.index, perm_df["mean"], xerr=perm_df["std"], color="darkorange", alpha=0.85)
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_title("2. Permutation Importance on Test Set (ROC-AUC drop)")
axes[1].set_xlabel("Decrease in Test ROC-AUC when feature is shuffled")

plt.tight_layout()
plt.show()

### 6.3 物理化学解读：对照 Yalkowsky 通用溶解度方程

右图的置换重要性揭示了一个惊人的事实：一旦打乱 `MolLogP`，测试集 ROC-AUC 直接暴跌近 `0.25`；而单独打乱 `MolWt` 或 `HeavyAtomCount`，ROC-AUC 几乎纹丝不动（因为打乱其中一个时，模型还能靠另一个高度共线的特征以及 `MolLogP` 维持预测）！

为什么 `MolLogP` 会呈现如此绝对的统治地位？我们可以对照物理化学中著名的 **Yalkowsky 通用溶解度方程（General Solubility Equation, GSE）**：

$$\log S \approx 0.5 - \log P - 0.01 \cdot (T_m - 25)$$

一个分子要溶解到水中，必须跨越两道热力学关卡：
1. **溶剂化项（$-\log P$）**：分子脱离自身液相进入水相的自由能变化，直接由疏水性描述符 `MolLogP` 决定——这正是随机森林牢牢抓住的第一主因；
2. **晶格破坏项（$-0.01(T_m - 25)$）**：把分子从固体晶格里拆出来所需的能量，在实验上体现为**熔点 $T_m$**。

请注意：**熔点 $T_m$ 是三维晶体堆积（Crystal Packing）的宏观性质，很难仅凭单个分子的 2D 拓扑结构直接算准！** 我们的 10 个 RDKit 2D 描述符里没有 $T_m$，只能靠 `NumAromaticRings`、`RingCount` 和 `FractionCSP3` 间接地去猜分子刚性与晶格能。这就解释了为什么有些“对称性极高、晶格极稳定（俗称晶格如砖头，brick-dust）”的小分子，虽然 $\log P$ 不高，实际上却极难溶解，从而成为模型漏报的重灾区。

我们在**第五讲（特征工程与模型可解释性 SHAP）**中还会进一步展开：如何剔除冗余共线特征，以及如何用 SHAP 具体量化每个特征对**单个分子**预测结果的推拉贡献。

### 🏋️ 练习 02c-3：揪出模型“漏报”最严重的难溶分子并分析原因

在测试集中找出**实际上难溶（`y_test == 1`），但随机森林给出的难溶概率 `P(insoluble)` 最低**的前 6 个分子（即模型最确信它们可溶、漏报最严重的 6 个假阴性样本）：
1. 打印它们的名称（`name`）、真实 `logS`、`MolLogP` 和模型预测的 `P(insoluble)`；
2. 用 `Draw.MolsToGridImage` 画出这 6 个分子的结构式；
3. 结合上面的 Yalkowsky 方程或分子结构特点，思考为什么模型会误判它们。

In [ ]:
# 在下方填写你的代码：


---

## 小结：第二讲（监督学习）知识全景

通过 `02a`、`02b`、`02c` 三个 Notebook，我们已经完整走通了材料与化学表格数据监督学习的标准工作流：

| 阶段 | 核心工具与概念 | 关键避坑准则 |
|---|---|---|
| **1. 问题定义** | 连续数值预测 → **回归（Regression）**<br>离散类别筛查 → **分类（Classification）** | 分类时明确哪一类是重点拦截的正类（`1`），切勿只看 Accuracy |
| **2. 数据划分与验证** | `train_test_split`（分类加 `stratify=y`）<br>`KFold` / `cross_val_score` 五折交叉验证 | 绝不在训练集上评估性能；单次划分有随机波动，对比模型看交叉验证均值 ± 标准差 |
| **3. 特征预处理** | `make_pipeline(StandardScaler(), model)` | 距离/核/线性模型必须做标准化，且 `Scaler` 只能 `fit` 训练集（用 Pipeline 防泄漏） |
| **4. 分子表征对比** | 10 个物理化学描述符 vs Morgan 指纹（Bit / Count） | 光看子结构“有没有”（Bit FP）会丢失分子量大小信息；描述符 + 计数指纹互补效果最好 |
| **5. 调参与适用域** | `GridSearchCV` 网格搜索<br>Tanimoto 化学相似度适用域检查 | 测试集插值准不代表能向全新的大分子家族外推；预测前先检查与训练集的化学距离 |
| **6. 模型解释与误差分析** | 树分裂重要性（MDI）vs 置换重要性（Permutation）<br>对照物理化学先验（如 Yalkowsky 方程） | 高度共线特征会稀释树分裂重要性；结合物理方程找准特征盲区（如 2D 特征缺失晶格熔点项） |

### 下一讲预告

在第二讲中，每个分子都有明确的实验标签 $\log S$（**监督学习，Supervised Learning**）。但如果面对一个刚构建的、还没有任何实验测量数据的十万级分子库，我们该如何：
1. 把 2048 维的分子指纹投影到二维平面上，绘制直观的**化学空间地图（Chemical Space Map）**？
2. 自动把结构相似的分子归成几个家族（**聚类，Clustering**），或者挑出结构最奇特的异常分子？

这些没有标签 $y$、只靠特征矩阵 $X$ 探索数据内在结构的任务，就是下一讲 **`03 - Unsupervised Learning`（无监督学习：降维、聚类与化学空间探索）** 的核心内容。